## Measuring the Clock Quality Statistic

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
import os
import csv
import numpy as np
import matplotlib.pylab as plt
from astropy.io import fits
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname("star_functions.py"), '..')))
import star_functions as nana
from astropy.table import Table
import os

In [ ]:
#for saving pngs later
save_dir = "best_clock_plots"
os.makedirs(save_dir, exist_ok=True)

### Get relevant data from best clocks fits file
#### Note: Features are not relevant here because there is a diff val for M

In [ ]:
fn = "/Users/natsuki/Projects/hogg_research/hoggnation/oscillator_catalog/good_parents_fit.fits"
with fits.open(fn) as hdu_list:
    print(hdu_list.info())
    good_parents_data = hdu_list[1].data
    star_ids = good_parents_data['star_id']
    parent_mode_ids = good_parents_data['parent_mode_id']
    features = good_parents_data["features"]
    freqs = good_parents_data["refined_frequency"]


# pairs = list(zip(star_ids, parent_mode_ids))
# print(pairs[:10])  # sanity check


### sanity check

In [ ]:
print(parent_mode_ids[:5], parent_mode_ids.shape)

### compute the clock quality from parent mode

In [ ]:
def find_clock_quality(parent_mode_id, plotting = False, three_points = False):
    '''
     compute_clock_quality()
    Computes the clock quality statistic dI/dt = freq^3 * [df/dphi]^2 / s^2
    for a given parent mode id.

    ## Inputs:
    - `parent_mode_id`: identifier for the mode

    ## Returns:
    - `dIdt`: the clock quality statistic
    - `refined_freq`: the refined frequency used
    - `M`: harmonic order used
    - `s2`: median squared residual 
    '''

    # get the frequency and kicid of this mode from parent view from the db
    conn = nana.get_db_connection()
    cursor = conn.cursor()
    cursor.execute(f'SELECT star_id, frequency FROM mode WHERE mode_id = {parent_mode_id}')
    star_id, freq = cursor.fetchone()
    
    # get the lightcurve for this kicid
    lc, delta_f, sampling_time, exptime = nana.get_kepler_data(star_id)
    t_fit, flux_fit, weight_fit = nana.mask_vals(lc, star_id)
    
    # choose M
    nyquist = 1.0 / (2 * sampling_time)    
    M = np.round(0.5 * nyquist / freq).astype(int)
    if M > 64:
        M = 64
    assert M != 0, f"find_M() returned 0 for freq={freq}, star_id={star_id}"
    
    # get chi2 values at 3 frequencies near the parent frequency
    epsilon = 0.5 * delta_f
    freqs = np.array([-epsilon * 3, 0, epsilon * 3]) + freq
    chi2s = np.zeros_like(freqs)
    
    for i in range(len(freqs)):
        chi2s[i] = nana.integral_chi_squared(2 * np.pi * freqs[i], t_fit, flux_fit, weight_fit, exptime, M=M)

    assert np.argmin(chi2s) == 1, (
    f"Central point is not the minimum, "
    f"chi2s={chi2s}"
    )
    
    refined_freq, refined_chi2 = nana.find_min_and_refine(freqs, chi2s, star_id)

    # Perform fourier series fit to degree M
    om = refined_freq * 2 * np.pi
    A = nana.integral_design_matrix(t_fit, om, exptime, M=M)
    coeffs = nana.weighted_least_squares(A, flux_fit, weight_fit)
    

    #plot chi square minimum if interested
    if plotting == True:
        fine_freqs = np.linspace(freq - 5 * epsilon, freq + 5 * epsilon, 100)
        fine_chi2s = np.array([nana.integral_chi_squared(2 * np.pi * f, t_fit, flux_fit, weight_fit, exptime, M=M) for f in fine_freqs])
        fig = plt.figure()
        plt.plot(fine_freqs, fine_chi2s, label='$\\chi^2$', color='black')
        plt.plot(freqs, chi2s, 'o', color='pink', label='Sampled points')
        plt.axvline(freq, color='red', alpha=0.5, label='Original frequency')
        plt.axvline(refined_freq, color='blue', alpha=0.5, label='Refined frequency')
        plt.title(f"$\\chi^2$ minimum — {star_id}")
        plt.xlabel("Frequency $d^{-1}$")
        plt.ylabel("$\\chi^2$")
        plt.legend()
        plt.show()
        plt.close(fig)

    #Take the mean square or median square residual
    f_fit = A @ coeffs
    resid = flux_fit - f_fit
    s2 = np.median(resid ** 2)

    assert s2 != 0, (f"s2 is equal to 0, can't have denom as 0")

    #Take the deriavtive of the fourier fit wrt to phase
    # new coeffs become b1, -a1, 2b2, -2a2, 3b3, -3a3, ..., Mb_M, -Ma_M

    dfdphi_sq = 0
    for m in range(1, M + 1):
        a = coeffs[2*m - 1]
        b = coeffs[2*m]
        dfdphi_sq = dfdphi_sq + m**2 * (a**2 + b**2)


    #compute clock quality!!!
    dIdt = refined_freq ** 3 * dfdphi_sq / s2

    #plot periodogram if interested
    if plotting == True: 
        fig = plt.figure()
        over_sampling = 3
        df, f_maxC = delta_f/over_sampling, (3 / (2*sampling_time))
        f_min = over_sampling * df
        freq_full, power_full = nana.get_periodogram(f_min, f_maxC, df, lc, star_id)
        conn = nana.get_db_connection()
        cursor = conn.cursor()
        cursor.execute(f"SELECT parent_frequency FROM parent_modes WHERE star_id = '{star_id}'")
        parent_freqs = [row[0] for row in cursor.fetchall()]
        cursor.execute(f"SELECT frequency FROM mode WHERE star_id = '{star_id}' AND parent_mode_id IS NOT NULL")
        child_freqs = [row[0] for row in cursor.fetchall()]
        conn.close()
        plt.plot(freq_full, power_full, 'k.', markersize=1, alpha=0.5)
        plt.xlabel("Frequency (1/day)")
        plt.ylabel("Power")
        plt.semilogy()
        plt.title(f"Log Periodogram of {star_id}")
        for pf in parent_freqs:
            plt.axvline(pf, color='red', alpha=0.5, lw=0.75)
        for cf in child_freqs:
            plt.axvline(cf, color='red', alpha=0.2, lw=0.5)
        plt.axvline(refined_freq, color='navy', alpha=0.2, lw=2.5, zorder = 5)
        fig.savefig(os.path.join(save_dir, f"{star_id}_periodogram.png"), dpi=300, bbox_inches='tight')
        plt.show()
        plt.close(fig)

    #plot phase fold if interested
    if plotting == True:
        fig = plt.figure()
        theta = np.linspace(0, 4 * np.pi, 1000)
        yplot0 = np.zeros_like(theta)
        for m in range(1, M + 1):
            a = coeffs[2*m - 1]
            b = coeffs[2*m]
            yplot0 += a * np.cos(m * theta) + b * np.sin(m * theta)
        yplot0 += coeffs[0]

        phase = (om * t_fit) % (4 * np.pi)

        fig, ax = plt.subplots(figsize=(6, 5))
        ax.plot(phase, flux_fit, alpha=0.1, markersize=2, marker='.', linestyle='none', color='k', rasterized=True)
        ax.plot(theta, yplot0, color='r', alpha=0.6, zorder=3)
        ax.set_xlabel('Phase')
        ax.set_ylabel('Flux')
        ax.set_title(rf'{star_id}: freq = {refined_freq:.04f} $day^{{-1}}$, M = {M}, clock quality = {dIdt:.03f}')
        #
        fig.savefig(os.path.join(save_dir, f"{star_id}_folded.png"), dpi=300, bbox_inches='tight')
        plt.show()
        plt.close(fig)
            
            
    if three_points == True:
    # fine grid around refined_freq
        span = delta_f / 5 
        #fine frequncy grid, and the three particualr points
        fine_freqs = np.linspace(refined_freq - span, refined_freq + span, 15)
        three_freqs = np.array([refined_freq - (delta_f/10), refined_freq, refined_freq + (delta_f/10)])
        fine_dIdts = np.zeros(15)
        three_dIdts = np.zeros(3)
        
        for i, freq in enumerate(fine_freqs):
            om = freq * 2 * np.pi
            A = nana.integral_design_matrix(t_fit, om, exptime, M=M)
            coeffs = nana.weighted_least_squares(A, flux_fit, weight_fit)
            
            f_fit = A @ coeffs
            resid = flux_fit - f_fit
            s2 = np.median(resid ** 2)
            assert s2 != 0, f"s2 is 0 at freq={freq}, can't have denom as 0"
            
            dfdphi_sq = 0
            for m in range(1, M + 1):
                a = coeffs[2*m - 1]
                b = coeffs[2*m]
                dfdphi_sq += m**2 * (a**2 + b**2)
                
                fine_dIdts[i] = freq**3 * dfdphi_sq / s2

        
        for i, freq in enumerate(three_freqs):
            om = freq * 2 * np.pi
            A = nana.integral_design_matrix(t_fit, om, exptime, M=M)
            coeffs = nana.weighted_least_squares(A, flux_fit, weight_fit)
            
            f_fit = A @ coeffs
            resid = flux_fit - f_fit
            s2 = np.median(resid ** 2)
            assert s2 != 0, f"s2 is 0 at freq={freq}, can't have denom as 0"
            
            dfdphi_sq = 0
            for m in range(1, M + 1):
                a = coeffs[2*m - 1]
                b = coeffs[2*m]
                dfdphi_sq += m**2 * (a**2 + b**2)
                
                three_dIdts[i] = freq**3 * dfdphi_sq / s2
        
        # the three original points (refined_freq and its two neighbors), for highlightin
        
        
        fig = plt.figure()
        plt.plot(fine_freqs, fine_dIdts, '-', color='black', label='Clock quality')
        plt.plot(three_freqs, three_dIdts, 'o', color='hotpink', markersize=8, zorder=3, label='± 1/(observation time * 10)')
        plt.axvline(refined_freq, color='green', label='Refined frequency')
        plt.xlabel("Frequency $d^{-1}$")
        plt.ylabel("Clock quality dI/dt")
        plt.title(f"Clock quality = {dIdt:.3f}, freq = {refined_freq:.6f}, M = {m}, {star_id} ")
        plt.legend()
        plt.show()
        plt.close(fig)


            
    return dIdt, refined_freq, star_id, M

### testing the three points clock quality evaluation

In [ ]:
find_clock_quality(51, plotting = True, three_points = True)

### Making fits file

In [ ]:
# dIdts, refined_freqs, star_ids, Ms, good_parent_mode_ids = [], [], [], [], []

# for parent_id in parent_mode_ids:
#     try:
#         dIdt, refined_freq, star_id, M = clock_quality(parent_id)
#     except AssertionError as e:
#         print(f"Skipping parent_mode_id={parent_id}: {e}")
#         continue
#     except Exception as e:
#         print(f"Unexpected error for parent_mode_id={parent_id}: {e}")
#         continue

#     good_parent_mode_ids.append(parent_id)
#     dIdts.append(dIdt)
#     refined_freqs.append(refined_freq)
#     star_ids.append(star_id)
#     Ms.append(M)

# output_table = Table()
# output_table['parent_mode_id'] = good_parent_mode_ids
# output_table['star_id'] = star_ids
# output_table['clock_quality'] = dIdts
# output_table['refined_frequency'] = refined_freqs
# output_table['M'] = Ms
# output_table.write('clock_quality_stats.fits', overwrite=True)

In [ ]:
fn = "/Users/natsuki/Projects/hogg_research/hoggnation/oscillator_catalog/notebooks/clock_quality_stats.fits"
with fits.open(fn) as hdu_list:
    print(hdu_list.info())
    good_parents_data = hdu_list[1].data
    star_ids = good_parents_data['star_id']
    parent_mode_ids = good_parents_data['parent_mode_id']
    clock_quality = good_parents_data['clock_quality']

# pairs = list(zip(star_ids, parent_mode_ids, clock_quality))
# print(pairs[:10]) 
print(clock_quality[:10])

### sort and plot the highest clock qualities

In [ ]:
sind = np.argsort(-clock_quality)

star_ids, parent_mode_ids, clock_quality = star_ids[sind], parent_mode_ids[sind], clock_quality[sind]

grouped = list(zip(star_ids, parent_mode_ids, clock_quality))
print(grouped[:5])

### plotting highest clock quality examples

In [ ]:
high_cqs = np.array([33412, 36905, 68584, 109900, 76593])

for mode_id in high_cqs:
    find_clock_quality(mode_id)

### now let's put all phase and periodograms in a folder!!!

In [ ]:
#split into thirds
thirds = np.array_split(parent_mode_ids, 3)
first_third, second_third, third_third = thirds

print(len(first_third), len(second_third), len(third_third))

In [ ]:
for mode_id in first_third:
    try:
        dIdt, refined_freq, star_id, M = find_clock_quality(mode_id, plotting=True)
    except AssertionError as e:
        print(f"Skipping mode_id={mode_id}: {e}")
        continue
    except Exception as e:
        print(f"Unexpected error for mode_id={mode_id}: {e}")
        continue    

In [ ]:
#last star number 269
second_third = second_third[265:]
for i, mode_id in enumerate(second_third):
    try:
        dIdt, refined_freq, star_id, M = find_clock_quality(mode_id, plotting=True)
    except AssertionError as e:
        print(f"Skipping mode_id={mode_id}: {e}")
        continue
    except Exception as e:
        print(f"Unexpected error for mode_id={mode_id}: {e}")
        continue    
    print("star number", i)

In [ ]:
for mode_id in third_third:
    try:
        dIdt, refined_freq, star_id, M = find_clock_quality(mode_id, plotting=True)
    except AssertionError as e:
        print(f"Skipping mode_id={mode_id}: {e}")
        continue
    except Exception as e:
        print(f"Unexpected error for mode_id={mode_id}: {e}")
        continue    

### create funciton that gets parent mode ids from the star id

In [ ]:
## you have to run the opening fits file cell first

fn = "/Users/natsuki/Projects/hogg_research/hoggnation/oscillator_catalog/notebooks/clock_quality_stats.fits"
with fits.open(fn) as hdu_list:
    print(hdu_list.info())
    good_parents_data = hdu_list[1].data
    star_ids = good_parents_data['star_id']
    parent_mode_ids = good_parents_data['parent_mode_id']
    clock_quality = good_parents_data['clock_quality']
    
def find_parent_mode_ids(kic_id):
   

    mask = (np.char.strip(star_ids) == kic_id.strip())
    matching_mode_ids = parent_mode_ids[mask]
    matching_cqs = clock_quality[mask]

    if len(matching_mode_ids) == 0:
        print(f"No parent_mode_ids found for {kic_id}")

    return matching_mode_ids

In [ ]:
KIC007917485 poster child phi - m to check, but doesn't have a parent mode (not identified as good clock, didn't pass variance test)

In [ ]:
#poster_child = "KIC006780873"
poster_child = "KIC007917485"

### get the mode_ids because there aren't any parent mode ids

In [ ]:
def find_mode_ids_for_star(star_id):
    """
    Look up all mode_ids for a given star directly from the `mode` table,
    bypassing the variance > 1e-4 filter used in fit_fourier_series_to_good_parents.
    """
    conn = nana.get_db_connection()
    cursor = conn.cursor()
    cursor.execute(f"SELECT mode_id, frequency FROM mode WHERE star_id = '{star_id}'")
    rows = cursor.fetchall()
    conn.close()

    if len(rows) == 0:
        print(f"No modes found for {star_id}")
    return rows  # list of (mode_id, frequency) tuples

In [ ]:
#from paper primary modes are 15.3830026 (20157) , 20.2628968 (20160) , 14.8566285 (20163) -- KIC006780873
#from paper primary modes are 14.1876418 (198593) and 13.4363385 (198596) 
poster_child2 = "KIC006780873"
rows = find_mode_ids_for_star(poster_child2)
print(rows)

In [ ]:
find_clock_quality(198596, plotting = True, three_points = True)

In [ ]:
find_clock_quality(